In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "Data").exists():
    ROOT = ROOT / "ThucHanh"
sys.path.insert(0, str(ROOT))
from recommendation import CarRecommender, Rule, DEFAULT_WEIGHTS, run_evaluation as run, MODE_LABELS
print("Python:", sys.version.split()[0], "NumPy:", np.__version__, "Pandas:", pd.__version__)


Python: 3.12.12 NumPy: 1.26.4 Pandas: 3.0.3


## 1. Dữ liệu và tiền xử lý
Loại bản ghi trùng hoàn toàn, kiểm tra giá trị thiếu và phạm vi hợp lệ. Không loại các xe chỉ vì cùng tên: mỗi hàng khác thuộc tính vẫn là một trường hợp độc lập. `brand` được trích từ `name`; tên dòng xe giữ để hiển thị, `seller_type` giữ để tra cứu nhưng không tính điểm.

In [2]:
raw = pd.read_csv(ROOT / "data" / "CAR DETAILS FROM CAR DEKHO.csv")
print("Dữ liệu gốc:", raw.shape)
print("Trùng hoàn toàn:", int(raw.duplicated().sum()))
print("Hàng có giá trị thiếu:", int(raw.isna().any(axis=1).sum()))
raw.head()


Dữ liệu gốc: (4340, 8)
Trùng hoàn toàn: 763
Hàng có giá trị thiếu: 0


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## 2. Chọn tham số và đánh giá tái lập
Lấy 360 xe mục tiêu không trùng: 60 cho validation, 300 cho test (100 phiên/seed 42, 43, 44). Thử ba bộ trọng số và alpha thuộc {0, 0.3} trên validation; chốt theo Precision@5. Bộ khuyến nghị không được truy cập hồ sơ mong muốn ẩn; chỉ nhận xe mẫu và các phản hồi do mô phỏng người dùng lựa chọn.

Mỗi hồ sơ có các ràng buộc phù hợp đa thuộc tính. Người dùng kiểm tra Top-5 và dừng khi có ít nhất một xe thỏa tất cả yêu cầu; nếu chưa có, phê bình xe Top-1. Top-10 cuối cùng luôn giữ các điều kiện đã phê bình. Số vòng tối đa là 10, vòng khởi tạo được tính là 0.

In [3]:
metadata = run(ROOT / "results")
engine = CarRecommender(ROOT / "data" / "CAR DETAILS FROM CAR DEKHO.csv",
                        weights=metadata["selected"]["weights"], alpha=metadata["selected"]["alpha"])
engine.data.head()


CBR tĩnh completed
CBR + phê bình đơn completed
CBR + phê bình tổ hợp completed
CBR + phê bình động completed
    mode                 model  sessions  precision5  precision10  seed_p5_std  success_rate  cycles_all  cycles_success  similarity5  empty_rate  mean_latency_ms
  static              CBR tĩnh       300    0.170667     0.172667     0.029687      0.263333    0.000000        0.000000     0.835954         0.0         4.027979
  simple    CBR + phê bình đơn       300    0.470000     0.452000     0.023580      0.953333    2.596667        2.234266     0.889212         0.0        10.525007
compound CBR + phê bình tổ hợp       300    0.516000     0.494667     0.036166      0.993333    1.936667        1.882550     0.892015         0.0         8.665227
 dynamic   CBR + phê bình động       300    0.539333     0.519333     0.020133      0.993333    2.246667        2.194631     0.891480         0.0        14.223037
Selected: {'preset': 'mileage', 'alpha': 0.0, 'precision5': 0.6066666666666

,case_id,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner,brand
0,0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner,Maruti
1,1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner,Datsun
4,4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner,Honda


## 3. Hàm tương tự
Thuộc tính số dùng $sim_i=\max(0,1-|t_i-x_i|/R_i)$ với $R_i=\max_i-\min_i$; thuộc tính phân loại dùng so khớp chính xác. Điểm tổng hợp là trung bình có trọng số trên các thuộc tính đã khai báo. Đây là độ tương tự từ khoảng cách tuyệt đối chuẩn hóa, không gọi là Euclidean.

Chức năng bất đối xứng được hỗ trợ và được thử trên validation; cấu hình được chọn sử dụng alpha=0. Điểm chẩn đoán ASS@5 được tính với bộ trọng số cơ sở cố định và alpha=0 để các mô hình có cùng cách chấm.

In [4]:
print("Trọng số chọn từ validation:", engine.weights)
print("Alpha:", engine.alpha)
query = engine.item(1)
example = engine.data.iloc[[0, 2, 3, 4]].copy()
example["similarity"] = engine.similarity(query)[example.case_id.to_numpy()]
example[["name", "selling_price", "km_driven", "year", "fuel", "transmission", "owner", "brand", "similarity"]]


Trọng số chọn từ validation: {'selling_price': 0.25, 'km_driven': 0.35, 'year': 0.15, 'fuel': 0.08, 'transmission': 0.07, 'owner': 0.05, 'brand': 0.05}
Alpha: 0.0


,name,selling_price,km_driven,year,fuel,transmission,owner,brand,similarity
0,Maruti 800 AC,60000,70000,2007,Petrol,Manual,First Owner,Maruti,0.989210
2,Hyundai Verna 1.6 SX,600000,100000,2012,Diesel,Manual,First Owner,Hyundai,0.808427
3,Datsun RediGO T Option,250000,46000,2017,Petrol,Manual,First Owner,Datsun,0.891455
4,Honda Amaze VX i-DTEC,450000,141000,2014,Diesel,Manual,Second Owner,Honda,0.734145


## 4. Truy hồi và phê bình đơn hoặc tổ hợp
Điều kiện được tích lũy. Xe mẫu ban đầu được loại bằng `case_id`; thứ tự đồng điểm được cố định theo ID. Không quay về toàn bộ danh mục khi tính kết quả cuối.

In [5]:
current = engine.item(2)
initial = engine.rank(current, k=10, exclude=[2])
rules = [engine.critique_rule(current, "selling_price"), engine.critique_rule(current, "km_driven")]
after = engine.rank(current, rules, k=10, exclude=[2])
print("Xe mẫu:", current["name"], current["selling_price"], current["km_driven"])
print("Điều kiện:", [r.describe() for r in rules])
print("Số ứng viên:", int(engine.mask(rules, [2]).sum()))
after[["name", "selling_price", "km_driven", "year", "similarity"]]


Xe mẫu: Hyundai Verna 1.6 SX 600000 100000
Điều kiện: ['giá bán < 600,000', 'số kilomet < 100,000']
Số ứng viên: 1984


,name,selling_price,km_driven,year,similarity
2204,Hyundai Verna 1.6 SX,390000,90000,2012,0.989749
2318,Hyundai Verna 1.6 SX,480000,80000,2012,0.987943
2428,Hyundai Verna 1.6 SX CRDi (O),390000,90000,2011,0.984391
2820,Hyundai i20 Magna Optional 1.4 CRDi,350000,80000,2012,0.984283
2009,Hyundai i20 Asta (o) 1.4 CRDi (Diesel),440000,70000,2012,0.982478
886,Hyundai i20 Asta 1.4 CRDi,300000,90000,2013,0.981858
2281,Hyundai i20 1.4 CRDi Asta,400000,70000,2012,0.981352
2811,Hyundai Verna 1.6 SX CRDi (O),525000,71000,2013,0.979948
1814,Hyundai i20 1.4 CRDi Sportz,321000,81257,2011,0.978655
1301,Hyundai i20 Asta 1.4 CRDi,550000,90000,2015,0.978182


## 5. Phê bình động
Khai thác tập truy hồi tối đa 100 xe, sinh thay đổi đơn và tổ hợp hai thuộc tính. Support là tỷ lệ xe trong tập hiện tại thỏa đồng thời các thay đổi. Chỉ giữ các mẫu có ít nhất 2 xe, support từ 5% đến 95%; ưu tiên tổ hợp và support thấp còn khả thi. Nếu người dùng mô phỏng không chấp nhận gợi ý, dùng một phê bình đơn.

In [6]:
patterns = engine.dynamic_critiques(current, exclude=[2])
pd.DataFrame([{k:p[k] for k in ["label", "count", "pool_size", "support"]} for p in patterns])


,label,count,pool_size,support
0,"giá bán < 600,000 và năm sản xuất > 2,012",41,100,0.41
1,"giá bán < 600,000 và số kilomet < 100,000",57,100,0.57
2,"số kilomet < 100,000 và năm sản xuất > 2,012",63,100,0.63
3,"giá bán < 600,000",67,100,0.67
4,"năm sản xuất > 2,012",71,100,0.71
5,"số kilomet < 100,000",84,100,0.84


## 6. Xử lý kết quả rỗng
Truy hồi theo tương tự vẫn có xe gần nhất. Tập rỗng ở đây do bổ sung lọc cứng. Thuật toán đề xuất sửa điều kiện dựa trên danh mục, ưu tiên thay ít điều kiện nhất, mức vi phạm nhỏ nhất rồi tương tự cao nhất. Điều kiện `relaxable=False` luôn được bảo vệ. Chỉ áp dụng sau khi người dùng đồng ý, không gọi thuật toán này là QUICKXPLAIN.

In [7]:
impossible = [Rule("selling_price", "le", 10000), Rule("transmission", "eq", "Automatic", False)]
assert engine.rank(current, impossible).empty
proposals = engine.repair_proposals(current, impossible)
for i,p in enumerate(proposals, 1):
    print(i, p["changes"], "Số xe:", p["candidate_count"])
repaired = engine.rank(current, proposals[0]["rules"])
assert (repaired.transmission == "Automatic").all()
repaired[["name", "selling_price", "transmission", "similarity"]]


1 ['giá bán ≤ 10,000 → giá bán ≤ 79,000'] Số xe: 1
2 ['giá bán ≤ 10,000 → giá bán ≤ 85,000'] Số xe: 2
3 ['giá bán ≤ 10,000 → giá bán ≤ 95,000'] Số xe: 3


,name,selling_price,transmission,similarity
2974,Maruti Esteem AX,79000,Automatic,0.691957


## 7. Kết quả và giới hạn
Precision@K = số xe phù hợp trong Top-K chia K; vị trí thiếu được tính không phù hợp. Tỷ lệ thành công và số vòng được báo cáo riêng. Bộ stress 30 trường hợp tách khỏi 300 phiên test: 20 có thể sửa và 10 có yêu cầu bắt buộc không thể đáp ứng. Đây là kết quả người dùng mô phỏng, chưa phải khảo sát người dùng thật.

In [8]:
summary = pd.read_csv(ROOT / "results" / "summary.csv")
print("Chênh lệch P@5 động - tĩnh:", metadata["paired_p5_delta"])
print("Khoảng bootstrap 95%:", metadata["paired_p5_ci95"])
summary[["model", "precision5", "precision10", "success_rate", "cycles_all", "cycles_success", "similarity5"]]


Chênh lệch P@5 động - tĩnh: 0.36866666666666664
Khoảng bootstrap 95%: [0.33133333333333337, 0.4080166666666666]


,model,precision5,precision10,success_rate,cycles_all,cycles_success,similarity5
0,CBR tĩnh,0.170667,0.172667,0.263333,0.000000,0.000000,0.835954
1,CBR + phê bình đơn,0.470000,0.452000,0.953333,2.596667,2.234266,0.889212
2,CBR + phê bình tổ hợp,0.516000,0.494667,0.993333,1.936667,1.882550,0.892015
3,CBR + phê bình động,0.539333,0.519333,0.993333,2.246667,2.194631,0.891480


In [9]:
pd.read_csv(ROOT / "results" / "by_scenario.csv")


,mode,scenario,precision5,precision10,success_rate,cycles_all
0,static,ngan_sach,0.223333,0.215000,0.300000,0.000000
1,static,it_km,0.290000,0.293333,0.450000,0.000000
2,static,doi_moi,0.170000,0.190000,0.266667,0.000000
3,static,ket_hop,0.130000,0.121667,0.200000,0.000000
4,static,thuong_hieu,0.040000,0.043333,0.100000,0.000000
5,simple,ngan_sach,0.533333,0.530000,1.000000,2.033333
6,simple,it_km,0.543333,0.531667,0.966667,1.833333
7,simple,doi_moi,0.443333,0.441667,1.000000,1.983333
8,simple,ket_hop,0.470000,0.413333,0.916667,3.200000
9,simple,thuong_hieu,0.360000,0.343333,0.883333,3.933333


In [10]:
from recommendation import run_figures as draw_figures
draw_figures()


Exported 5 figures to D:\Khoa Học Dữ Liệu\Kì 7\Hệ Khuyến Nghị\Bài Tập Lớn Cá Nhân\ThucHanh\figures


![Precision@5 và Precision@10](figures/precision_comparison.png)

![Tỷ lệ thành công và số vòng](figures/success_and_cycles.png)

![Kết quả theo kịch bản](figures/scenario_precision.png)

## 8. Chạy giao diện và kiểm tra
Từ terminal trong thư mục project:
```bash
python recommendation.py test
python Website/app.py
```
Mở http://127.0.0.1:8765. Chọn xe mẫu, áp dụng phê bình đơn/tổ hợp/động, thử yêu cầu giá tối đa 10000 để xem nới lỏng, quay lại và tải lịch sử JSON. Code, dữ liệu và kết quả chi tiết đi kèm để tái lập. Công cụ hỗ trợ AI được khai báo trong README và báo cáo.
